# Heart Disease Classifier — Feature Engineering, Model Training & Experiment Tracking

**Course:** Machine Learning Operations (MLOps) — AIMLCZG523, BITS Pilani WILP M.Tech. AIML
**Assignment:** Assignment 01
**Tasks covered:**
- Task 2 — Feature Engineering & Model Development [8 marks]
- Task 3 — Experiment Tracking (MLflow) [5 marks]
- Task 4 — Model Packaging & Reproducibility [7 marks]

## Objective
Build a reusable `sklearn.Pipeline` (preprocessing + classifier), train and tune three
candidate models (Logistic Regression, Random Forest, XGBoost) with `GridSearchCV` +
stratified cross-validation, log every run to MLflow, and persist the best-performing
pipeline as a single reusable artifact for the FastAPI serving layer.

## Design note
All of the actual logic lives in `src/features.py` and `src/train.py` as **importable,
unit-tested functions** (see `tests/test_features.py`) — this notebook is a thin,
readable orchestration layer that calls those functions and narrates the results. This
avoids duplicating pipeline logic between the notebook and the production training
script (`python src/train.py`, which is also what the CI/CD pipeline and Docker image
build against), which would otherwise risk the two drifting out of sync.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.features import build_full_pipeline, build_preprocessing_pipeline, split_features_target, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.train import (
    load_data, get_model_grid, evaluate_model, plot_confusion_matrix, plot_roc_curve,
    train_and_track_all_models, RANDOM_STATE, TEST_SIZE, CV_FOLDS,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
import os
os.chdir("..")   # run from repo root so relative paths (data/, models/, reports/) resolve
print("Working directory:", os.getcwd())


## 1. Feature Engineering Pipeline

`src/features.py` defines a `ColumnTransformer` with two branches:

| Branch | Columns | Imputation | Transform |
|---|---|---|---|
| Numeric | age, trestbps, chol, thalach, oldpeak | median | StandardScaler |
| Categorical | sex, cp, fbs, restecg, exang, slope, ca, thal | most_frequent | OneHotEncoder(handle_unknown="ignore") |

Wrapping this in a `Pipeline` together with the classifier means the **exact same**
preprocessing is applied at training time and at inference time (loaded from a single
joblib file by the FastAPI service) — no train/serve skew.


In [ ]:
df = load_data("data/processed/heart_disease_clean.csv")
X, y = split_features_target(df)
print(f"Numeric features: {NUMERIC_FEATURES}")
print(f"Categorical features: {CATEGORICAL_FEATURES}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

preprocessor = build_preprocessing_pipeline()
X_transformed = preprocessor.fit_transform(X)
print(f"Transformed feature matrix shape: {X_transformed.shape} (one-hot expansion of categoricals)")


## 2. Train / Test Split

An 80/20 stratified split (`stratify=y`) so both splits preserve the ~54/46 class balance
identified in the EDA notebook.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))


## 3. Model Candidates & Hyperparameter Grids

Three classifiers are tuned via `GridSearchCV` with 5-fold stratified cross-validation,
optimizing for ROC-AUC (robust to the mild class imbalance):

- **Logistic Regression** — fast, interpretable baseline (tuned: `C`, `penalty`, `solver`)
- **Random Forest** — captures non-linear feature interactions (tuned: `n_estimators`,
  `max_depth`, `min_samples_leaf`)
- **XGBoost** — gradient-boosted trees, typically strong on tabular data (tuned:
  `n_estimators`, `max_depth`, `learning_rate`)


In [ ]:
for name, (estimator, grid) in get_model_grid().items():
    n_combinations = 1
    for v in grid.values():
        n_combinations *= len(v)
    print(f"{name}: {estimator.__class__.__name__}, {n_combinations} grid combinations x {CV_FOLDS}-fold CV")


## 4. Train, Tune, Evaluate, and Log to MLflow

Calling `train_and_track_all_models()` from `src/train.py` runs the full loop for all
three models: `GridSearchCV.fit()` → evaluate on held-out test set → log params/metrics/
plots/model to MLflow → select the best model by test ROC-AUC → persist it to
`models/heart_disease_pipeline.joblib`.

Run `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the repo root and open
**http://127.0.0.1:5000** to browse every run's parameters, metrics, and artifacts
interactively (screenshot this for the assignment report).


In [ ]:
summary = train_and_track_all_models()
print("\nFinal best model:", summary["best_model"])


## 5. Model Comparison Table

In [ ]:
results_df = pd.DataFrame(summary["results"]).T
results_df = results_df[["cv_best_roc_auc", "accuracy", "precision", "recall", "f1_score", "roc_auc"]]
results_df = results_df.sort_values("roc_auc", ascending=False)
results_df.style.background_gradient(cmap="Greens", subset=["roc_auc"]).format("{:.4f}")


**Interpretation:** Logistic Regression achieved the highest held-out test ROC-AUC
despite being the simplest model — a reasonable outcome on a small (303-row), largely
linearly-separable clinical dataset where the strongest predictors (`cp`, `thalach`,
`oldpeak`, `exang`, see EDA notebook) have fairly direct, monotonic relationships with the
target. The tree-based models (Random Forest, XGBoost) are close behind and remain useful
because they are more robust to any non-linear interactions not captured by the linear
baseline, and are logged to MLflow for comparison even though they weren't selected as
the final model.


## 6. Best Model's Confusion Matrix & ROC Curve (already logged to MLflow as artifacts)

In [ ]:
from IPython.display import Image, display
best_name = summary["best_model"]
display(Image(filename=f"reports/figures/confusion_matrix_{best_name}.png"))
display(Image(filename=f"reports/figures/roc_curve_{best_name}.png"))


## 7. Reproducibility Artifact Check

Confirm the final packaged model artifact (Task 4 deliverable) exists and can be
reloaded and used for inference identically to how `api/main.py` loads it.


In [ ]:
import joblib
pipeline = joblib.load("models/heart_disease_pipeline.joblib")
print(pipeline)

sample_patient = X_test.iloc[[0]]
pred = pipeline.predict(sample_patient)[0]
proba = pipeline.predict_proba(sample_patient)[0]
print(f"\nSample prediction: class={pred}, P(no disease)={proba[0]:.4f}, P(disease)={proba[1]:.4f}")
print(f"True label for this sample: {y_test.iloc[0]}")


## 8. Summary

- ✅ Preprocessing pipeline (imputation + scaling + one-hot encoding) built with
  `ColumnTransformer` inside a single `sklearn.Pipeline`.
- ✅ 3 classification models trained and tuned with `GridSearchCV` + 5-fold stratified CV.
- ✅ Evaluated with accuracy, precision, recall, F1, and ROC-AUC on a held-out test set.
- ✅ Every run logged to MLflow: hyperparameters, CV score, test metrics, confusion
  matrix + ROC plots, and the fitted pipeline as an MLflow model artifact.
- ✅ Best model (by test ROC-AUC) persisted to `models/heart_disease_pipeline.joblib` —
  a single, self-contained, reusable artifact loaded directly by `api/main.py`.

Next: see `notebooks/03_inference_demo.ipynb` for a standalone inference walkthrough, or
`api/main.py` for the production FastAPI service that wraps this exact artifact.
